# Compile finalized per-block saccades

Load saccade CSVs written by the preprocessing GUI **Saccades** tab
(`block/analysis/saccades/`) across a paper registry, QC them, and optionally
hand off to `PaperContext` / write a combined pickle for the flexible paper tool.

**Prerequisites:** each block finalized via the Saccades tab (or equivalent
`write_finalized_saccades`). Blocks without `analysis/saccades/` are reported
and skipped.

See also: `flexible_paper_figures_tool.ipynb` (`USE_FINALIZED_SACCADES=True`).

## 0. Setup

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO = Path.cwd()
if not (REPO / "src" / "eye_tracking_system_tools").is_dir():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "eye_tracking_system_tools").is_dir():
            REPO = p
            break

sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("MPLCONFIGDIR", str(REPO / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

from eye_tracking_system_tools.analysis.block_registry import load_registry
from eye_tracking_system_tools.analysis.event_cache import save_event_cache
from eye_tracking_system_tools.analysis.export_meta import load_params_yaml
from eye_tracking_system_tools.analysis.paper_gui import PaperContext, block_qc_table
from eye_tracking_system_tools.analysis.pipeline import build_event_tables
from eye_tracking_system_tools.analysis.pixel_calibration import has_pixel_calibration
from eye_tracking_system_tools.analysis.run_layout import resolve_run_dir
from eye_tracking_system_tools.analysis.saccade_export import (
    detection_params_fingerprint,
    has_finalized_saccades,
    read_finalized_saccades,
    saccades_dir,
)

print("REPO:", REPO)

## 1. Paths & registry

In [ ]:
# --- edit me ---
REGISTRY = REPO / "configs" / "paper_blocks.yaml"
# REGISTRY = REPO / "configs" / "paper_blocks_custom.yaml"
# REGISTRY = REPO / "configs" / "sample_blocks.yaml"
PARAMS = REPO / "configs" / "analysis_params.yaml"
TAG = "compiled_saccades"
# ---------------

params = load_params_yaml(PARAMS)
specs = load_registry(REGISTRY)
run = resolve_run_dir(REPO / "outputs", tag=TAG, prefix="paper")
print("REGISTRY:", REGISTRY)
print("blocks:", len(specs))
print("run_dir:", run.run_dir)

## 2. Inventory: which blocks have finalized saccades?

In [ ]:
rows = []
for spec in specs:
    has = has_finalized_saccades(spec.block_path)
    row = {
        "block_key": spec.block_key,
        "has_finalized": has,
        "has_pix_size": has_pixel_calibration(spec.block_path),
        "saccades_dir": str(saccades_dir(spec.block_path)),
    }
    if has:
        fin = read_finalized_saccades(spec.block_path)
        row["n_events"] = len(fin.all_saccades)
        row["n_synced"] = len(fin.synced)
        row["n_monocular"] = len(fin.non_synced)
        row["params_fp"] = detection_params_fingerprint(fin.params)
        row["source"] = fin.source
        row["isi_mean_ms"] = fin.summary.get("isi_mean_ms")
    rows.append(row)

inventory = pd.DataFrame(rows)
display(inventory)
missing = inventory.loc[~inventory["has_finalized"], "block_key"].tolist()
print(f"finalized: {inventory['has_finalized'].sum()} / {len(inventory)}")
if missing:
    print("missing:", ", ".join(missing))

## 3. Build EventTables (prefer finalized CSVs)

`build_event_tables(..., prefer_finalized=True)` loads `analysis/saccades/` when
present and falls back to on-the-fly detection otherwise.

In [ ]:
tables = build_event_tables(
    specs,
    params=params,
    keep_traces=False,
    prefer_finalized=True,
)
print(
    f"blocks={len(tables.blocks)}  all_saccades={len(tables.all_saccades)}  "
    f"synced_rows={len(tables.synced)}  non_synced={len(tables.non_synced)}"
)

qc = block_qc_table(tables)
display(qc)

cache_path = save_event_cache(tables, run.metadata_dir, specs=specs)
print("cached →", cache_path)

ctx = PaperContext(
    tables,
    run.run_dir,
    registry_path=REGISTRY,
    params_path=PARAMS,
)
print("PaperContext ready — open flexible_paper_figures_tool.ipynb or build figures here.")